# SISTEMA DE RECONOCIMIENTO DE HABLA EN ESPAÑOL.   

- X es una emisión de de una *secuencia de unidades*
- Asigna modelos modelos ocultos de Markov a unidades.
- Unimos *todos los estados de cada modelo*, mediante las probabilidades  $P(m^P | m^{P-1})$ formando un modelo único gigante.
- Buscamos $\hat{z} / P(x, \hat{z}) = \max\limits_{ \forall z} P(x, z)$


![image.png](attachment:image.png)

> Cada de esos globitos serian modelos de palabras, imaginate que existen muchas... esto no escala.


## Modelos de unidades.

Como planteamos el modelo de unidades?
- Definir cuál va a ser la unidad -> **Palabras**  -> La prob. entre palabras tiene su sentido concreto (lenguage)
- Definir cuál va a ser el modelo oculto de Markov de cada unidad.

> ESto está bien cuando las palabras estan entre 20-100 palabras **Pero** a veces se inventan palabras o aparecen nuevas palabras y esto no funciona.

**Modelos de subunidades**  -> Se pueden generar cualquier unidad con subreglas (convenciones linguisticas).



![image2.png](attachment:image.png)

> Es mucho mas ventajoso este enfoque que entrenar palabras.

Entonces tengo que definirme una secuencia de palabras y fonemas

(Son audios translados a secuencias de coeficientes Ceptrum)
![image3.png](image3.png)

Luego se debe conseguir un modelo para cada fonema donde $P(m_H^{P}|m_H^{P-1})$ es el modelo del lenguage, nota que rtiene que ver con la transcicion de la palabra, es algo que trata del lenguage y no del sonido.

![image4.png](image4.png)

## Objetivos del proyecto
* implementar un sistema de reconocimiento automático de habla en idioma español de aproximadamente 3000 palabras.
* Implementar un sistema de reconocimiento de gramática finita de alrededor de 30 palabras para ser usado en tiempo real.
Se implementará un baseline con la base de datos latino40 con modelos de monofonos. Luego se implementará una gramática finita para usarla en tiempo real.



![image5.png](image5.png)

    Nosotros vamos a usar el modelo de monofonos, podriamos usar trifonos pero es mas complica aunque dá mejores resultados.



### Creación del directorio del proyecto
Vamos a crear nuestro directorio **`proyecto_hc`** y su estructura.



#### 1. Crear los directorios principales
```bash
mkdir config datos etc lm log modelos rec scripts
```



#### 2. Copiar los archivos requeridos

Desde la carpeta `/home/cestien/Documentos/`:
```bash
cp /home/cestien/Documentos/config.hcopy ./config/
cp /home/cestien/Documentos/go.mfclist ./scripts/
```


#### 3. Crear subdirectorios dentro de `datos`
```bash
mkdir datos/mfc
mkdir datos/wav
```

#### 4. Visualizar el árbol de carpetas
```bash
tree -L 2
```

Salida:

    .
    ├── config
    │   └── config.hcopy
    ├── datos
    │   ├── mfc
    │   └── wav
    ├── etc
    ├── lm
    ├── log
    ├── modelos
    ├── rec
    └── scripts
        └── go.mfclist

    10 directories, 2 files


Ahora lo que debemos hacer es pasar los archivos de audio a la carpeta `datos/wav` y los archivos de texto a la carpeta `datos/mfc`.

Pero los archivos wav estan en otro directorio!, asi que vamos a enlazarlos (no queremos copiarlos):

Se sabe que tenemos dos directorios de train y devel, queremos quer sean los subdirectorios en datos de train y test respectivamente, usamos en el enlace simbolico (-s) con el comando ln.
```bash
bfuentes@habla:~/proyecto_hc$ ls /dbase/latino40/
devel  doc  leer  train

bfuentes@habla:~/proyecto_hc/datos$ cd wav/
bfuentes@habla:~/proyecto_hc/datos/wav$ ln -s /dbase/latino40/train/ train
bfuentes@habla:~/proyecto_hc/datos/wav$ ln -s /dbase/latino40/train/ test
bfuentes@habla:~/proyecto_hc/datos/wav$ ls
test  train
```

Y se puede chequear que los archivos estan en la carpeta `datos/wav`:

```bash
bfuentes@habla:~/proyecto_hc/datos/wav$ tree -L 2
.
├── test -> /dbase/latino40/train/
└── train -> /dbase/latino40/train/

2 directories, 0 files
```


Ahora, se deben crear las carpetas train y test en **mfc** para que el sistema pueda crear los archivos de mfc.

```bash
bfuentes@habla:~/proyecto_hc/datos/mfc$ mkdir test train
```

Se tiene que tener lo siguiente!
```bash
bfuentes@habla:~/proyecto_hc$ tree -L 3
.
├── config
│   └── config.hcopy
├── datos
│   ├── mfc
│   │   ├── test
│   │   └── train
│   └── wav
│       ├── test -> /dbase/latino40/train/
│       └── train -> /dbase/latino40/train/
├── etc
├── lm
├── log
├── modelos
├── rec
└── scripts
    └── go.mfclist

14 directories, 2 files
```

Muy bien, ahora veamos que hay en config.hcopy

```bash
bfuentes@habla:~/proyecto_hc$ cat config/config.hcopy
 # Coding parameters
 TARGETKIND = MFCC_0
 #TARGETKIND = MFCC_0_D_A
 TARGETRATE = 100000.0
 SAVECOMPRESSED = T
 SAVEWITHCRC = T
 WINDOWSIZE = 250000.0
 USEHAMMING = T
 PREEMCOEF = 0.97
 NUMCHANS = 26
 CEPLIFTER = 22
 NUMCEPS = 12
 ENORMALISE = F
 SOURCEFORMAT = NIST
```
Son parametros del experimento, alguien ya los obtuvo pero de no tenerlos habria que probar hasta llegar a esos parametros.

- **TARGETKIND**: Es el tipo de cepstrum que se va a usar, en este caso es MFCC_0 es el C0 como coeficiente de energia $c_i = \sqrt{2/N} \sum_{j=1}^{N} m_j cos(\frac{\pi i}{N} (j-0.5))$ / $m_j = \log(4(j))$ entonces $c_0 = \sqrt{2/N} \sum\limits_{j=1}^{NumChans} \log(4 (j))$.

-

- **NUMCHANS**:  
- **NUMCEPS**:   Es el numero de cepstrum en escala de mel que se van a extraer, seria el liftering.
- **CEPTILFER**: ES mas bien un ponderado de los coeficientes para que los coeficientes que no sean comparables lo sean.
- **WINDOWSZISE**: Es el tamaño de la ventana de la transformada  en nano segundos (aca son 25ms).
- **TARGETRATE**: Es la tasa de muestreo del audio (10msec).
- **SOURCEFORMAT**: Es el formato del audio (NIST).

![image6.png](image6.png)

Vamos a usar HCOPY

![image7.png](image7.png)

> Si no anda el Hcopy usar:
>
>    ```bash
>    bfuentes@habla:~/proyecto_hc/datos$ PATH=$PATH:/usr/local/speechapp/htk/bin/
>    ```



**go.mfclist**: es un archivo shell  que va leer un directorio y mientra a leyendo va creando subdirectorios, basicamente automatiza el proceso de creacion de mfc.

> Agregar permisos de ejecucion al archivo go.mfclist:

```bash
bfuentes@habla:~/proyecto_hc/scripts$ chmod +x go.mfclist
```

Se debe ejecutar desde la carpeta **datos**.


```bash
HCopy -T 1 -C ../config/config.hcopy -S genmfc.train
```

- T: nivel de trace o nivel de salida.
- C: archivo de configuracion.
- S: archivo de entrada (Script file).

```bash
bfuentes@habla:~/proyecto_hc/datos$ HList -F NIST -h -e 10 wav/train/af01/af01_001.wav
-------------------- Source: wav/train/af01/af01_001.wav --------------------
  Sample Bytes:  2        Sample Kind:   WAVEFORM
  Num Comps:     1        Sample Period: 62.5 us
  Num Samples:   76801    File Format:   NIST
------------------------------ Samples: 0->11 -------------------------------
    0:     396    384    376    381    372    362    364    354    358    374
   10:     360    351
------------------------------------ END ------------------------------------
```

## clase 3

**Repaso**

Supongamos que tebemis modelo de O y S.


O-> {mujo, sigmajo, Ao}
S-> {mujs, sigmajs, As}

que ya los tenemos.

y luego emisiones

x1,x2,x3----xT.

y tenemos oso, todo lo que va cambiar es la matriz A.

c_ [1 0 0, diag{Ao, As, Ao}]

Ai con matris de columnas de ceros a derecha.

Eso es entrenameinto embebido, se entrenan globitos en j que pueden ser iguales ej: A2o = Ano va a ser lo mismo.

Se hace un unico paso E para todo .

* Hay que explicar entrenamiento embebido y en otro apartado aparte el tp7.



pausas.



..... __ dijo

queremos agregar un sp entre espacoos.


barcos [sp] bla bla


model T, un estado que se puede saltear, el modelo de sp el modelo de silencio.


el sp es un modelo corto de ida, y el modelo largo es el que tiene ida y vuelta.

HHEd === Htk Hmmdefs Editor.


Hoy hay que traerse de la pc host el ultimo modelo que creamos hmm3 nos lo tenemos que traer a la pc usando sftp y lo editamos en pc para volver a poner en directorio 4.


> cp hmm3/macros hmm4/

> sftp user@ipblabla (en otra terminal)

buscar:
"sil" y aparece ~h "sil" y aparece otro sil igual.

~h "sil"
....
~h "sil"

todo eso lo copio abajo y lo llamo ~h "sp"

y le cambiamos a <NUMSTATE> 3
<STATE> 2 todo eso lo borro y lo mismo con 4.


En la matriz <TRANSP> 3

y saco los estados que yo borré.
Ahora solo quedó el estado 3 :/ borro columnas.

luego lo guardo y mando al directorio 4, hmm4 y le hago put.

> cp /home/cestine/Documentos/sil.hed ./

Luego en etc queremos editar
> nano monophones+sil

editamos luego de
sil
sp
.... guardamos.
tenemos que tener dos, monophones+sil y monophoes+sil+sp

y ahora si se puede ejeturar
> HHED -T 1 -H hmm4/macros -H hmm4/hmmdefs -M hmm sil.hed ../etc/monophones+sil+sp






para dejarlo en background el Hvite.


dsgfadsgdrf &   (no pongamos el trace)


para salir poner un exit!